# Apache Beam Lab 2: Pipeline Fundamentals - Solution

## Overview
This notebook contains the solution for Lab 2: Pipeline Fundamentals.

## Solution Implementation

In [ ]:
import apache_beam as beam

def basic_pipeline():
    with beam.Pipeline() as pipeline:
        pass

print("Pipeline concept: A Pipeline encapsulates your entire data processing workflow")

## PCollection Creation

In [ ]:
def create_pcollections():
    with beam.Pipeline() as pipeline:
        numbers = pipeline | 'Create numbers' >> beam.Create([1, 2, 3, 4, 5])
        words = pipeline | 'Create words' >> beam.Create(['hello', 'world', 'beam'])
        
        import pandas as pd
        sales_file = '/home/ramdov/projects/beam-code-practice/data/sample_sales.csv'
        if __import__('os').path.exists(sales_file):
            df = pd.read_csv(sales_file)
            data = df.to_dict('records')
            sales = pipeline | 'Create sales' >> beam.Create(data)

print("PCollection concept: A distributed dataset that your pipeline processes")

## Basic Transforms

In [ ]:
def demonstrate_transforms():
    with beam.Pipeline() as pipeline:
        numbers = pipeline | 'Create' >> beam.Create([1, 2, 3, 4, 5])
        
        doubled = numbers | 'Double' >> beam.Map(lambda x: x * 2)
        evens = numbers | 'Filter even' >> beam.Filter(lambda x: x % 2 == 0)
        total = numbers | 'Sum' >> beam.CombineGlobally(sum)
        
        doubled | 'Print doubled' >> beam.Map(lambda x: print(f"Doubled: {x}"))
        evens | 'Print evens' >> beam.Map(lambda x: print(f"Even: {x}"))
        total | 'Print total' >> beam.Map(lambda x: print(f"Total: {x}"))

print("Transforms: Operations that process PCollections to produce new PCollections")

## Exercise 1: Word Count Pipeline

In [ ]:
def word_count_pipeline():
    import re
    
    with beam.Pipeline() as pipeline:
        (
            pipeline
            | 'Create lines' >> beam.Create([
                'Hello world',
                'Hello Apache Beam',
                'World of data processing',
                'Beam makes data processing easy'
            ])
            | 'Split into words' >> beam.FlatMap(lambda x: x.split())
            | 'Normalize' >> beam.Map(lambda x: re.sub(r'[^a-zA-Z]', '', x.lower()))
            | 'Filter empty' >> beam.Filter(lambda x: x)
            | 'Count words' >> beam.Map(lambda x: (x, 1))
            | 'Group and sum' >> beam.CombinePerKey(sum)
            | 'Format' >> beam.Map(lambda x: f"{x[0]}: {x[1]}")
            | 'Print' >> beam.Map(print)
        )

print("Running word count pipeline...")
word_count_pipeline()

## Exercise 2: Sales Analysis Pipeline

In [ ]:
import pandas as pd
import os

def sales_analysis_pipeline():
    sales_file = '/home/ramdov/projects/beam-code-practice/data/sample_sales.csv'
    
    if not os.path.exists(sales_file):
        print("Sales data not found. Run Lab 0 first.")
        return
    
    df = pd.read_csv(sales_file)
    data = df.to_dict('records')
    
    with beam.Pipeline() as pipeline:
        total_revenue = (
            pipeline
            | 'Create sales data' >> beam.Create(data)
            | 'Calculate transaction total' >> beam.Map(
                lambda x: x['quantity'] * x['price'])
            | 'Sum revenue' >> beam.CombineGlobally(sum)
        )
        
        avg_transaction = (
            pipeline
            | 'Create data for avg' >> beam.Create(data)
            | 'Calculate totals' >> beam.Map(
                lambda x: x['quantity'] * x['price'])
            | 'Calculate average' >> beam.CombineGlobally(
                lambda values: sum(values) / len(values) if values else 0)
        )
        
        customer_counts = (
            pipeline
            | 'Create data for counts' >> beam.Create(data)
            | 'Extract customer' >> beam.Map(lambda x: (x['customer_id'], 1))
            | 'Count per customer' >> beam.CombinePerKey(sum)
        )
        
        total_revenue | 'Print revenue' >> beam.Map(
            lambda x: print(f"Total Revenue: ${x:.2f}"))
        avg_transaction | 'Print average' >> beam.Map(
            lambda x: print(f"Average Transaction: ${x:.2f}"))
        customer_counts | 'Print counts' >> beam.Map(
            lambda x: print(f"Customer {x[0]}: {x[1]} transactions"))

print("Running sales analysis pipeline...")
sales_analysis_pipeline()

## Exercise 3: Complex Pipeline with Multiple Branches

In [ ]:
def complex_pipeline():
    sales_file = '/home/ramdov/projects/beam-code-practice/data/sample_sales.csv'
    
    if not os.path.exists(sales_file):
        print("Sales data not found. Run Lab 0 first.")
        return
    
    df = pd.read_csv(sales_file)
    data = df.to_dict('records')
    
    with beam.Pipeline() as pipeline:
        sales_data = pipeline | 'Create sales data' >> beam.Create(data)
        
        (
            sales_data
            | 'Branch 1: Calculate revenue' >> beam.Map(
                lambda x: x['quantity'] * x['price'])
            | 'Branch 1: Sum revenue' >> beam.CombineGlobally(sum)
            | 'Branch 1: Print revenue' >> beam.Map(
                lambda x: print(f"Total Revenue: ${x:.2f}"))
        )
        
        (
            sales_data
            | 'Branch 2: Extract product' >> beam.Map(
                lambda x: (x['product_id'], x['quantity'] * x['price']))
            | 'Branch 2: Sum by product' >> beam.CombinePerKey(sum)
            | 'Branch 2: Format' >> beam.Map(
                lambda x: print(f"Product {x[0]}: ${x[1]:.2f}"))
        )
        
        (
            sales_data
            | 'Branch 3: Extract customer' >> beam.Map(
                lambda x: (x['customer_id'], 1))
            | 'Branch 3: Count customers' >> beam.CombinePerKey(sum)
            | 'Branch 3: Format' >> beam.Map(
                lambda x: print(f"Customer {x[0]}: {x[1]} transactions"))
        )

print("Running complex pipeline with multiple branches...")
complex_pipeline()

## Summary

This solution demonstrates:
- Pipeline creation and structure
- PCollection creation from various sources
- Basic transforms (Map, Filter, Combine)
- Word count implementation
- Sales data analysis
- Multi-branch pipeline architecture

These fundamental concepts form the foundation for more advanced Apache Beam patterns.